# 01 — Why Data Validation?

## Real-world problem

You receive `orders.csv` every day from an upstream e-commerce system. The file loads successfully with pandas, but that does **not** prove that the data is trustworthy.

In this notebook we first inspect the data **without Pandera**. The goal is to feel the pain that a data contract solves.


## 1. Load the raw dataset

Run the cell below. Do not fix any values yet.

In [7]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

DATA_PATH = ROOT / 'data' / 'raw' / 'orders.csv'
df = pd.read_csv(DATA_PATH)
df

,order_id,customer_id,product_id,quantity,unit_price,discount,total,status,order_date,internal_note
0,1001,C001,P001,2,1200.0,0.10,2160.0,paid,2026-08-01,priority customer
1,1002,C002,P002,1,25.0,0.00,25.0,shipped,2026-08-02,NaN
2,1003,C003,P003,0,80.0,0.00,0.0,paid,2026-08-03,zero quantity
3,1004,C004,P004,3,-50.0,0.10,-135.0,pending,2026-08-04,negative price
4,1005,C005,P005,1,300.0,1.20,-60.0,paid,2026-08-05,discount over 100 percent
5,1006,C006,P006,2,75.0,0.10,100.0,shipped,2026-08-06,wrong total
6,1007,C007,P007,2,40.0,0.00,80.0,UNKNOWN,2026-08-07,invalid status
7,1008,C008,P008,1,50.0,0.00,50.0,cancelled,2026-08-08,valid row
8,1008,C009,P009,1,70.0,0.00,70.0,paid,2026-08-09,duplicate order id
9,1010,C010,P010,NaN,90.0,0.10,81.0,paid,2026-08-10,missing quantity


### Think before continuing

Look at the table manually. Write down at least five suspicious values before using any automated inspection.

## 2. What did pandas infer?


In [8]:
df.dtypes

order_id           int64
customer_id          str
product_id           str
quantity             str
unit_price       float64
discount         float64
total            float64
status               str
order_date           str
internal_note        str
dtype: object

### Questions

- Did pandas infer `quantity` as an integer?
- Why might a single bad value affect the dtype of an entire column?
- Does a successful `read_csv` mean the dataset is valid?

## 3. Missing values


In [9]:
df.isna().sum()

order_id         0
customer_id      1
product_id       0
quantity         1
unit_price       0
discount         0
total            0
status           0
order_date       0
internal_note    1
dtype: int64

## 4. Duplicate identifiers


In [10]:
df[df['order_id'].duplicated(keep=False)].sort_values('order_id')

,order_id,customer_id,product_id,quantity,unit_price,discount,total,status,order_date,internal_note
7,1008,C008,P008,1,50.0,0.0,50.0,cancelled,2026-08-08,valid row
8,1008,C009,P009,1,70.0,0.0,70.0,paid,2026-08-09,duplicate order id


## 5. Unexpected categories


In [11]:
df['status'].value_counts(dropna=False)

status
paid         8
shipped      3
pending      1
UNKNOWN      1
cancelled    1
refunded     1
Name: count, dtype: int64

## 6. Manual business-rule checking gets messy

The business says:

```text
total = unit_price * quantity * (1 - discount)
```

But `quantity` currently contains a non-numeric value, so even this simple check requires cleaning or coercion decisions first.

In [12]:
quantity_numeric = pd.to_numeric(df['quantity'], errors='coerce')
expected_total = df['unit_price'] * quantity_numeric * (1 - df['discount'])

manual_total_check = pd.DataFrame({
    'order_id': df['order_id'],
    'actual_total': df['total'],
    'expected_total': expected_total,
    'matches': df['total'].round(2) == expected_total.round(2),
})
manual_total_check

,order_id,actual_total,expected_total,matches
0,1001,2160.0,2160.0,True
1,1002,25.0,25.0,True
2,1003,0.0,0.0,True
3,1004,-135.0,-135.0,True
4,1005,-60.0,-60.0,True
5,1006,100.0,135.0,False
6,1007,80.0,80.0,True
7,1008,50.0,50.0,True
8,1008,70.0,70.0,True
9,1010,81.0,NaN,False


## 7. Reflection

Answer these before moving on:

1. Which problems are about **structure or dtype**?
2. Which problems are about **individual values**?
3. Which problems require comparing **multiple columns**?
4. Which bad values should be rejected, and which might reasonably be coerced?
5. How maintainable would it be to write all validation manually with pandas?

Next, read `docs/01_order_data_contract.md` and begin translating the contract into `OrderSchema`.

## Interview checkpoint

**Question:** If `pd.read_csv()` successfully loads a file, why is that not enough evidence that the data is valid?

Write your answer in your own words before continuing.